# F1-fast: QLoRA-дообучение русскоязычной VLM

Первая обучаемая версия проекта. Базовая `Qwen2.5-VL-3B-Instruct` загружается в 4-битном виде; изменяются только LoRA-адаптеры. Для быстрой проверяемой итерации из 12 000 примеров `LLaVA-Instruct-ru` отбираются 2 000, сохраняя доли типов. GQA-ru/MMBench-ru не используются в обучении.

**Конфигурация F1-fast:** rank=8, 1 эпоха, batch size=1, накопление градиента=8. На RTX 4060 ожидаемое время — около 1,5–2 часов.

In [ ]:
import os
import json
from io import BytesIO
from pathlib import Path

import pandas as pd
import requests
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration,
    Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from urllib3.exceptions import InsecureRequestWarning

# Обычная загрузка Hugging Face работает стабильнее Xet в текущей сети.
os.environ['HF_HUB_DISABLE_XET'] = '1'
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
SUBSET_PATH = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
MODEL_DIR = ROOT / 'models' / 'f1_fast_qlora_adapter'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'f1_fast_qlora'
RESULTS_DIR = ROOT / 'results'
for directory in (IMAGE_CACHE, MODEL_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SEED = 42
TRAINING_EXAMPLES = 2_000
MAX_LENGTH = 1024
assert torch.cuda.is_available(), 'Нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Подготовка обучающей выборки

Берём уже сохранённый в EDA список строк. Это делает F1 воспроизводимой: даже при повторном запуске состав 12 000 примеров не меняется.

In [ ]:
subset = pd.read_csv(SUBSET_PATH)
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))

def parse_record(record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return question.replace('<image>\n', '').strip(), answer.strip(), record['image']

train_rows = []
for row in subset[['row_number', 'type']].itertuples(index=False):
    question, answer, image_path = parse_record(raw_records[int(row.row_number)])
    train_rows.append({'type': row.type, 'question': question, 'answer': answer, 'image_path': image_path})

full_train_df = pd.DataFrame(train_rows)
# Стратифицированная подвыборка: сохраняет долю conversation и complex_reasoning.
train_df = (full_train_df.groupby('type', group_keys=False)
            .apply(lambda group: group.sample(
                n=round(TRAINING_EXAMPLES * len(group) / len(full_train_df)), random_state=SEED
            ))
            .reset_index(drop=True))
train_df = train_df.sample(n=TRAINING_EXAMPLES, random_state=SEED).reset_index(drop=True)
assert len(train_df) == TRAINING_EXAMPLES
display(train_df.head(3))
display(train_df['type'].value_counts().rename_axis('type').reset_index(name='examples'))

In [ ]:
def image_url(image_path):
    _, split, filename = image_path.split('/')
    return f'https://images.cocodataset.org/{split}/{filename}'

def load_image(image_path):
    filename = Path(image_path).name
    local_path = IMAGE_CACHE / filename
    if not local_path.exists():
        response = requests.get(image_url(image_path), timeout=60, verify=False)
        response.raise_for_status()
        local_path.write_bytes(response.content)
    return Image.open(local_path).convert('RGB')

class RussianVlmDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {'question': row.question, 'answer': row.answer, 'image_path': row.image_path}

dataset = RussianVlmDataset(train_df)
example = dataset[0]
print(example['question'])
print(example['answer'][:180] + '...')
display(load_image(example['image_path']))

## 2. Предварительная загрузка изображений

Выполните эту ячейку **после отключения VPN** и дождитесь окончания. Она скачивает изображения до обучения, поэтому во время `trainer.train()` не будет сетевых пауз. Ячейку можно безопасно перезапускать: существующие файлы пропускаются, а неполные загрузки сохраняются во временный файл.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

MAX_WORKERS = 8
image_paths = train_df['image_path'].drop_duplicates().tolist()

# Сначала проверяем сеть на одном файле. Если сертификат подменяется прокси,
# переходим к загрузке публичных COCO-изображений без SSL-проверки.
probe_url = image_url(image_paths[0])
try:
    probe = requests.get(probe_url, timeout=(10, 30), verify=True)
    probe.raise_for_status()
    VERIFY_SSL = True
    print('SSL-проверка прошла: изображения будут скачиваться в обычном защищённом режиме.')
except requests.exceptions.SSLError as error:
    print('SSL-проверка не прошла (вероятно, сертификат подменяется прокси).')
    print('Для открытого архива COCO будет использован режим verify=False.')
    probe = requests.get(probe_url, timeout=(10, 30), verify=False)
    probe.raise_for_status()
    VERIFY_SSL = False
except Exception as error:
    raise RuntimeError(f'Нет доступа к COCO до начала загрузки: {error}') from error

def download_one(image_path):
    filename = Path(image_path).name
    target = IMAGE_CACHE / filename
    temporary = target.with_suffix('.part')
    if target.exists() and target.stat().st_size > 0:
        return 'already_cached', filename
    if temporary.exists():
        temporary.unlink()
    for attempt in range(1, 4):
        try:
            response = requests.get(image_url(image_path), timeout=(10, 45), verify=VERIFY_SSL)
            response.raise_for_status()
            temporary.write_bytes(response.content)
            temporary.replace(target)
            return 'downloaded', filename
        except Exception as error:
            if temporary.exists():
                temporary.unlink()
            if attempt == 3:
                return 'failed', f'{filename}: {error}'
    return 'failed', filename

stats = {'already_cached': 0, 'downloaded': 0, 'failed': 0}
failed_files = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_one, path) for path in image_paths]
    for completed, future in enumerate(tqdm(as_completed(futures), total=len(futures), desc='Загрузка COCO'), start=1):
        status, value = future.result()
        stats[status] += 1
        if status == 'failed':
            failed_files.append(value)
        if completed <= 3 or completed % 25 == 0:
            print(f'[{completed}/{len(image_paths)}] последний файл: {value}; статистика: {stats}')

print('Итог:', stats)
if failed_files:
    print('Не удалось скачать файлов:', len(failed_files))
    print('Первые ошибки:', failed_files[:5])
else:
    print('Все изображения для F1 готовы. Можно переходить к разделу 4 и запускать обучение.')

## 3. Модель и LoRA-адаптер

Видеомодель загружается в NF4. LoRA подключается к линейным слоям внимания и MLP языковой части; visual encoder остаётся замороженным. Ограничение разрешения снижает потребление видеопамяти и делает первый запуск реалистичным для 8 ГБ VRAM.

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization, device_map='auto', torch_dtype=torch.float16
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
class VlmDataCollator:
    def __call__(self, features):
        images = [load_image(item['image_path']) for item in features]
        full_texts, prompt_texts = [], []
        for item, image in zip(features, images):
            user_message = {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': item['question']},
            ]}
            full_messages = [user_message, {'role': 'assistant', 'content': [{'type': 'text', 'text': item['answer']}]}]
            full_texts.append(processor.apply_chat_template(full_messages, tokenize=False))
            prompt_texts.append(processor.apply_chat_template([user_message], tokenize=False, add_generation_prompt=True))

        batch = processor(
            text=full_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
)
        prompt_batch = processor(
            text=prompt_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
)
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100
        prompt_lengths = prompt_batch['attention_mask'].sum(dim=1).tolist()
        for i, prompt_length in enumerate(prompt_lengths):
            labels[i, :prompt_length] = -100
        batch['labels'] = labels
        return batch

collator = VlmDataCollator()
test_batch = collator([dataset[0]])
{name: tuple(value.shape) for name, value in test_batch.items()}

## 4. Запуск F1

Чекпойнты и итоговый адаптер не добавляются в Git: они предназначены для DVC/DagsHub. После окончания обучения следующий ноутбук повторит baseline-оценку и сравнит B0 с F1.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=8,
    logging_steps=5,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=dataset, data_collator=collator,
)
print(f'Примерное число оптимизационных шагов: {len(dataset) // training_args.gradient_accumulation_steps}')
trainer.train()

model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
(RESULTS_DIR / 'f1_fast_training_config.json').write_text(json.dumps({
    'base_model': MODEL_ID, 'examples': len(dataset), 'lora_rank': 8,
    'epochs': 1, 'learning_rate': 2e-4, 'seed': SEED,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Адаптер сохранён в: {MODEL_DIR}')